# ASR base: Whisper small fine-tuned en Shipibo-Konibo (dataset v1.0)

Modelo de referencia (*baseline*) entrenado solo con audio real pareado. Los experimentos con audio
sintético y pseudoetiquetado se comparan contra este.

**Reproducible:** versiones de librerías y del modelo fijadas, semilla fija, `config.json` con el
sha256 del dataset, y la partición guardada en `split_v1.csv`, que es la misma para todos los experimentos.

**Reanudable:** si Colab se desconecta, ejecuta todo de nuevo: el entrenamiento sigue desde el último checkpoint.

| partición | qué es |
|---|---|
| `train` | resto de SK001 + la mitad de los grupos de hablante de YouTube y workshop |
| `dev` | 5% de SK001: elige el mejor checkpoint |
| `test_sk001` | 10% de SK001: mismo hablante que train, frases no vistas |
| `test_otros` | la otra mitad de los grupos de YouTube y workshop: **voces que el modelo nunca oyó** |

Ninguna frase de dev/test aparece en train ni en `shp_apto_tts.txt`, así que el audio sintético
del TTS no contamina la evaluación.

In [ ]:
#@title 1. Parámetros
#@markdown **Datos** — ID o enlace de compartir ("Cualquier persona con el enlace"). Vacío = usa la ruta de Drive.
DATASET_ID = ""  #@param {type:"string"}
TEXTO_TTS_ID = ""  #@param {type:"string"}
DRIVE_DATASET = "/content/drive/MyDrive/TESIS/asr/dataset_v1.0.zip"  #@param {type:"string"}
DRIVE_TEXTO_TTS = "/content/drive/MyDrive/TESIS/modelo_tts/shp_apto_tts.txt"  #@param {type:"string"}
#@markdown Carpeta del experimento. `split_v1.csv` se guarda en su carpeta padre (compartido entre experimentos).
SALIDA = "/content/drive/MyDrive/TESIS/asr/whisper_small_base"  #@param {type:"string"}

#@markdown **Modelo** (revisión fija de Hugging Face)
MODELO = "openai/whisper-small"
REVISION = "973afd24965f72e36ca33b3055d56a652f456b4d"
IDIOMA = "spanish"   # Whisper no tiene shipibo: se usa el token de español (misma grafía latina)

#@markdown **Partición**
SEED = 1234  #@param {type:"integer"}
FRAC_DEV = 0.05
FRAC_TEST = 0.10
FRAC_OTROS = 0.5     # fracción de los minutos de YouTube y de workshop que va a test_otros

#@markdown **Entrenamiento**
LR = 1e-5  #@param {type:"number"}
BATCH = 16  #@param {type:"integer"}
ACUM = 2  #@param {type:"integer"}
WARMUP = 200  #@param {type:"integer"}
MAX_STEPS = 2000  #@param {type:"integer"}
EVAL_CADA = 100  #@param {type:"integer"}
PACIENCIA = 5  #@param {type:"integer"}
SPECAUGMENT = True  #@param {type:"boolean"}
EVAL_ZERO_SHOT = True  #@param {type:"boolean"}

SHA_DATASET_PROYECTO = "174b61329123a19a4ed9d09b4d8cb229eb7989cddc9d63d9888859c906993a5d"
SHA_TEXTO_PROYECTO = "9ee9d32f2ef77b8298594fd8a2ddf266eaae3ea3460de4e5598ebf3f53ba1527"

## 2. GPU y Drive

In [ ]:
import torch
if not torch.cuda.is_available():
    raise SystemExit("No hay GPU: Entorno de ejecución > Cambiar tipo de entorno > GPU (T4)")
print("GPU:", torch.cuda.get_device_name(0))

DRIVE_OK = False
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_OK = True
except Exception as e:
    print(f"No se pudo montar Drive ({type(e).__name__}: {e})")
if not DRIVE_OK and SALIDA.startswith("/content/drive"):
    SALIDA = "/content/asr_salida/" + SALIDA.rstrip("/").split("/")[-1]
    print(f"AVISO: sin Drive se guarda en {SALIDA}: se BORRA si Colab se desconecta. "
          "Descárgalo al terminar (celda 10).")

## 3. Librerías (versiones fijas)

In [ ]:
!pip install -q transformers==5.17.0 accelerate==1.15.0 jiwer==4.0.0 gdown

## 4. Núcleo (probado localmente antes de subirlo)

In [ ]:
%%writefile /content/asr_core.py
# ---- nucleo del ASR: se prueba localmente y se copia tal cual al notebook ----
import hashlib, re, unicodedata
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import torch

SR = 16000
FUENTE_SK = "menendez_gomez_2025"


# ============================ TEXTO ============================
def normalizar(t):
    """Misma normalizacion que build_dataset_v1.py (text_norm): NFC, minusculas, sin puntuacion.
    Se aplica tambien a lo que predice Whisper, que tiende a poner mayusculas y puntos."""
    t = unicodedata.normalize("NFC", str(t)).lower()
    t = re.sub(r"[¡!¿?.,;:…\"“”«»()\[\]{}—–\-_/\\]", " ", t)
    t = t.replace("'", " ").replace("’", " ")
    return re.sub(r"\s+", " ", t).strip()


def sha256(ruta):
    h = hashlib.sha256()
    with open(ruta, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()


# ============================ SPLIT ============================
def hacer_split(meta, lineas_tts, seed, frac_dev, frac_test, frac_otros):
    """Particion fija para todos los experimentos de la tesis.

    - test_otros: grupos de hablante ENTEROS de youtube y workshop (voces que el modelo no oye).
    - dev / test_sk001: clips de SK001 cuyo texto NO aparece en el txt del TTS, para que los
      experimentos con audio sintetico no vean las frases de evaluacion.
    - Un texto de evaluacion nunca queda tambien en train (se saca de train: 'fuera_texto').
    """
    m = meta.sort_values("id").reset_index(drop=True).copy()
    m["split"] = "train"
    rng = np.random.RandomState(seed)

    # 1) hablantes no vistos
    for fuente in sorted(set(m.source) - {FUENTE_SK}):
        d = m[m.source == fuente].groupby("speaker_group").duration_s.sum()
        grupos = sorted(d.index)
        rng.shuffle(grupos)
        acum, elegidos = 0.0, []
        for g in grupos:
            if acum >= frac_otros * d.sum():
                break
            elegidos.append(g)
            acum += d[g]
        m.loc[m.speaker_group.isin(elegidos), "split"] = "test_otros"

    # 2) SK001: textos que no esten contenidos en ninguna linea del TTS
    blob = "\n".join(f" {normalizar(l)} " for l in lineas_tts if l.strip())
    sk = m[m.source == FUENTE_SK]
    elegibles = sorted(t for t in set(sk.text_norm) if f" {t} " not in blob)
    rng.shuffle(elegibles)
    n_test, n_dev = round(frac_test * len(sk)), round(frac_dev * len(sk))
    cuenta = sk.text_norm.value_counts()
    usados = {"test_sk001": 0, "dev": 0}
    for t in elegibles:
        destino = "test_sk001" if usados["test_sk001"] < n_test else "dev" if usados["dev"] < n_dev else None
        if destino is None:
            break
        m.loc[(m.source == FUENTE_SK) & (m.text_norm == t), "split"] = destino
        usados[destino] += int(cuenta[t])

    # 3) sin fuga por texto: evaluacion con texto en el TTS, o train con texto de evaluacion
    evaluacion = m.split != "train"
    en_tts = m.text_norm.map(lambda t: f" {t} " in blob)
    m.loc[evaluacion & en_tts, "split"] = "fuera_tts"
    textos_eval = set(m.loc[m.split.isin(["dev", "test_sk001", "test_otros"]), "text_norm"])
    m.loc[(m.split == "train") & m.text_norm.isin(textos_eval), "split"] = "fuera_texto"
    return m


def resumen_split(m):
    r = m.groupby("split").agg(clips=("id", "size"), minutos=("duration_s", lambda s: round(s.sum() / 60, 1)),
                               hablantes=("speaker_group", "nunique"))
    return r.reindex([s for s in ["train", "dev", "test_sk001", "test_otros", "fuera_tts", "fuera_texto"]
                      if s in r.index])


# ============================ DATOS PARA WHISPER ============================
class ClipsASR(torch.utils.data.Dataset):
    def __init__(self, df, raiz, processor):
        self.filas = list(df[["file_name", "text_norm"]].itertuples(index=False))
        self.raiz, self.p = Path(raiz), processor

    def __len__(self):
        return len(self.filas)

    def __getitem__(self, i):
        archivo, texto = self.filas[i]
        y, sr = sf.read(self.raiz / archivo, dtype="float32")
        assert sr == SR, f"{archivo}: {sr} Hz"
        return {"input_features": self.p.feature_extractor(y, sampling_rate=SR).input_features[0],
                "labels": self.p.tokenizer(texto).input_ids}


@dataclass
class Collator:
    processor: object
    decoder_start_token_id: int

    def __call__(self, feats):
        batch = self.processor.feature_extractor.pad(
            [{"input_features": f["input_features"]} for f in feats], return_tensors="pt")
        lab = self.processor.tokenizer.pad([{"input_ids": f["labels"]} for f in feats], return_tensors="pt")
        labels = lab["input_ids"].masked_fill(lab["attention_mask"].ne(1), -100)
        if (labels[:, 0] == self.decoder_start_token_id).all():   # el modelo lo agrega solo
            labels = labels[:, 1:]
        batch["labels"] = labels
        return batch


# ============================ METRICAS ============================
def wer_cer(refs, hyps):
    import jiwer
    pares = [(r, h) for r, h in zip(refs, hyps) if r]            # jiwer no acepta referencias vacias
    r, h = [p[0] for p in pares], [p[1] for p in pares]
    return 100 * jiwer.wer(r, h), 100 * jiwer.cer(r, h)


def metricas_trainer(processor):
    tok = processor.tokenizer

    def f(pred):
        ids, lab = pred.predictions, pred.label_ids
        lab = np.where(lab == -100, tok.pad_token_id, lab)
        ids = np.where(ids == -100, tok.pad_token_id, ids)
        hyps = [normalizar(t) for t in tok.batch_decode(ids, skip_special_tokens=True)]
        refs = [normalizar(t) for t in tok.batch_decode(lab, skip_special_tokens=True)]
        w, c = wer_cer(refs, hyps)
        return {"wer": round(w, 2), "cer": round(c, 2)}
    return f


def wer_con_ic(pred, seed=0, n=1000):
    """WER corpus (no promedio por clip) e IC 95% por bootstrap sobre clips."""
    import jiwer
    p = pred[pred.text_norm.str.len() > 0]
    err, pal = [], []
    for a, b in zip(p.text_norm, p.prediccion):
        o = jiwer.process_words(a, b)
        err.append(o.substitutions + o.deletions + o.insertions)
        pal.append(o.substitutions + o.deletions + o.hits)
    err, pal = np.array(err), np.array(pal)
    rng = np.random.RandomState(seed)
    k = rng.randint(0, len(p), size=(n, len(p)))
    boot = 100 * err[k].sum(1) / pal[k].sum(1)
    return dict(wer=round(float(100 * err.sum() / pal.sum()), 2), ic95=[round(float(x), 2) for x in np.percentile(boot, [2.5, 97.5])],
                cer=round(100 * jiwer.cer(list(p.text_norm), list(p.prediccion)), 2), clips=int(len(p)),
                palabras=int(pal.sum()))


@torch.no_grad()
def transcribir(model, processor, df, raiz, device, batch=16, max_tokens=128, num_beams=1):
    """Transcribe df y devuelve una copia con la prediccion y el error por clip."""
    import jiwer
    model.eval()
    dtype = next(model.parameters()).dtype
    salida = []
    for i in range(0, len(df), batch):
        trozo = df.iloc[i:i + batch]
        audios = [sf.read(Path(raiz) / f, dtype="float32")[0] for f in trozo.file_name]
        x = processor.feature_extractor(audios, sampling_rate=SR, return_tensors="pt").input_features
        ids = model.generate(x.to(device, dtype), language="spanish", task="transcribe",
                             max_new_tokens=max_tokens, num_beams=num_beams)
        salida += processor.tokenizer.batch_decode(ids, skip_special_tokens=True)
    r = df[["id", "file_name", "source", "speaker_group", "duration_s", "text_norm"]].copy()
    r["prediccion"] = [normalizar(t) for t in salida]
    r["wer_clip"] = [round(100 * jiwer.wer(a, b), 1) if a else None for a, b in zip(r.text_norm, r.prediccion)]
    return r

## 5. Datos y partición

In [ ]:
import sys, importlib, importlib.metadata, json, re, shutil, zipfile, platform
sys.path.insert(0, "/content")
import asr_core as C
importlib.reload(C)
import pandas as pd
from pathlib import Path


def id_de(v):
    v = v.strip()
    m = re.search(r"/d/([-\w]{20,})", v) or re.search(r"[?&]id=([-\w]{20,})", v)
    return m.group(1) if m else v


def obtener(valor, ruta_drive, que):
    if valor.strip():
        import gdown
        fid, carpeta = id_de(valor), Path("/content/descargas")
        carpeta.mkdir(parents=True, exist_ok=True)
        marca = carpeta / f".{fid}"
        if marca.exists() and (carpeta / marca.read_text().strip()).exists():
            return carpeta / marca.read_text().strip()
        print(f"descargando {que} ({fid}) ...")
        try:
            out = gdown.download(id=fid, output=str(carpeta) + "/", quiet=True)
        except Exception as e:
            out = None
            print(f"   {type(e).__name__}: {e}")
        if not out or not Path(out).exists():
            raise SystemExit(f"No pude descargar {que}. En Drive: clic derecho > Compartir > "
                             f"Acceso general: 'Cualquier persona con el enlace' (Lector).")
        marca.write_text(Path(out).name)
        return Path(out)
    p = Path(ruta_drive)
    if not p.exists():
        raise SystemExit(f"No encuentro {que} en {p}. Pon su ID en la celda 1 o corrige la ruta.")
    return p


ZIP = obtener(DATASET_ID, DRIVE_DATASET, "dataset_v1.0.zip")
TXT_TTS = obtener(TEXTO_TTS_ID, DRIVE_TEXTO_TTS, "shp_apto_tts.txt")
sha_zip, sha_txt = C.sha256(ZIP), C.sha256(TXT_TTS)
print("dataset :", ZIP.name, "= el del proyecto" if sha_zip == SHA_DATASET_PROYECTO else f"DISTINTO al del proyecto ({sha_zip[:12]})")
print("texto TTS:", TXT_TTS.name, "= el del proyecto" if sha_txt == SHA_TEXTO_PROYECTO else f"DISTINTO al del proyecto ({sha_txt[:12]})")

# descomprimir en el disco local de Colab (leer 3 mil wavs desde Drive es muy lento)
DATOS = Path("/content/datos") / sha_zip[:12]
if not DATOS.exists():
    tmp = DATOS.with_name(DATOS.name + ".tmp")
    shutil.rmtree(tmp, ignore_errors=True)
    with zipfile.ZipFile(ZIP) as z:
        z.extractall(tmp)
    tmp.rename(DATOS)
RAIZ = next(DATOS.rglob("metadata.csv")).parent
meta = pd.read_csv(RAIZ / "metadata.csv")
print(f"{len(meta)} clips, {meta.duration_s.sum() / 3600:.2f} h")

lineas_tts = TXT_TTS.read_text(encoding="utf-8-sig").splitlines()
split = C.hacer_split(meta, lineas_tts, SEED, FRAC_DEV, FRAC_TEST, FRAC_OTROS)

out = Path(SALIDA)
out.mkdir(parents=True, exist_ok=True)
F_SPLIT = out.parent / "split_v1.csv"
if F_SPLIT.exists():   # la partición es de la tesis, no del experimento: nunca se reescribe
    previo = pd.read_csv(F_SPLIT).set_index("id").split
    if not previo.equals(split.set_index("id").split.loc[previo.index]) or len(previo) != len(split):
        raise SystemExit(f"{F_SPLIT} no coincide con la partición calculada ahora "
                         "(¿otro dataset, texto TTS o SEED?). No la mezcles: revisa la celda 1.")
    print(f"partición: la de {F_SPLIT} (coincide)")
else:
    split[["id", "split", "source", "speaker_group", "duration_s"]].to_csv(F_SPLIT, index=False)
    print(f"partición guardada en {F_SPLIT}")
print(C.resumen_split(split).to_string())
print("\ntest_otros:", ", ".join(sorted(split[split.split == "test_otros"].speaker_group.unique())))

# config de la corrida; si se reanuda con otros parámetros clave, se detiene
import transformers, accelerate, jiwer, numpy as np
cfg = {k: v for k, v in globals().items() if k.isupper() and isinstance(v, (int, float, str, bool))}
cfg.update(sha256_dataset=sha_zip, sha256_texto_tts=sha_txt, python=platform.python_version(),
           torch=torch.__version__, cuda=torch.version.cuda, gpu=torch.cuda.get_device_name(0),
           transformers=transformers.__version__, accelerate=accelerate.__version__,
           jiwer=importlib.metadata.version("jiwer"), numpy=np.__version__)
f_cfg = out / "config.json"
if f_cfg.exists():
    viejo = json.loads(f_cfg.read_text())
    for k in ("MODELO", "REVISION", "sha256_dataset", "SEED", "LR", "BATCH", "ACUM", "WARMUP",
              "MAX_STEPS", "EVAL_CADA", "SPECAUGMENT"):
        if viejo.get(k) != cfg.get(k):
            raise SystemExit(f"'{k}' cambió respecto de la corrida en {out} ({viejo.get(k)} -> {cfg.get(k)}). "
                             "Para otro experimento usa otra carpeta SALIDA.")
f_cfg.write_text(json.dumps(cfg, indent=2, ensure_ascii=False))

## 6. Modelo

In [ ]:
from transformers import WhisperForConditionalGeneration, WhisperProcessor, set_seed

set_seed(SEED)
processor = WhisperProcessor.from_pretrained(MODELO, revision=REVISION, language=IDIOMA, task="transcribe")
model = WhisperForConditionalGeneration.from_pretrained(MODELO, revision=REVISION)
model.generation_config.language = IDIOMA
model.generation_config.task = "transcribe"
model.generation_config.forced_decoder_ids = None
model.config.use_cache = False               # incompatible con gradient checkpointing
if SPECAUGMENT:                              # enmascara tiempo/frecuencia al entrenar: ayuda con pocos datos
    model.config.apply_spec_augment = True
    model.config.mask_time_prob = 0.05
    model.config.mask_feature_prob = 0.05

device = torch.device("cuda")
train_df = split[split.split == "train"]
dev_df = split[split.split == "dev"]
train_ds = C.ClipsASR(train_df, RAIZ, processor)
dev_ds = C.ClipsASR(dev_df, RAIZ, processor)
collator = C.Collator(processor, model.config.decoder_start_token_id)
print(f"train {len(train_ds)} clips | dev {len(dev_ds)} clips | "
      f"{MAX_STEPS * BATCH * ACUM / len(train_ds):.1f} épocas como máximo")

## 7. Punto de partida: Whisper sin entrenar (*zero-shot*)

Solo una vez por carpeta; sirve de fila de referencia en la tabla de resultados.

In [ ]:
EVAL = {s: split[split.split == s] for s in ("test_sk001", "test_otros")}
f_zs = out / "resultados_zero_shot.json"
if EVAL_ZERO_SHOT and not f_zs.exists():
    model.to(device)
    res = {}
    for nombre, df in EVAL.items():
        pred = C.transcribir(model, processor, df, RAIZ, device)
        pred.to_csv(out / f"pred_zero_shot_{nombre}.csv", index=False)
        res[nombre] = C.wer_con_ic(pred)
        print(nombre, res[nombre])
    f_zs.write_text(json.dumps(res, indent=2))
elif f_zs.exists():
    print(f_zs.read_text())

## 8. Entrenamiento (reanudable)

In [ ]:
from transformers import EarlyStoppingCallback, Seq2SeqTrainer, Seq2SeqTrainingArguments
from transformers.trainer_utils import get_last_checkpoint

bf16 = torch.cuda.is_bf16_supported()        # A100/L4: bf16 | T4: fp16
args = Seq2SeqTrainingArguments(
    output_dir=str(out / "checkpoints"),
    per_device_train_batch_size=BATCH, gradient_accumulation_steps=ACUM,
    learning_rate=LR, warmup_steps=WARMUP, max_steps=MAX_STEPS, lr_scheduler_type="linear",
    gradient_checkpointing=True, fp16=not bf16, bf16=bf16,
    eval_strategy="steps", eval_steps=EVAL_CADA, save_steps=EVAL_CADA, save_total_limit=2,
    load_best_model_at_end=True, metric_for_best_model="wer", greater_is_better=False,
    per_device_eval_batch_size=BATCH, predict_with_generate=True, generation_max_length=128,
    logging_steps=25, report_to="none", seed=SEED, data_seed=SEED, dataloader_num_workers=2,
)
trainer = Seq2SeqTrainer(
    model=model, args=args, train_dataset=train_ds, eval_dataset=dev_ds, data_collator=collator,
    compute_metrics=C.metricas_trainer(processor), processing_class=processor.feature_extractor,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=PACIENCIA)],
)
FINAL = out / "modelo_final"
if (FINAL / "config.json").exists():
    print(f"Ya entrenado: {FINAL}. Bórralo si quieres reentrenar en esta carpeta.")
    model = WhisperForConditionalGeneration.from_pretrained(FINAL)
else:
    ultimo = get_last_checkpoint(args.output_dir) if Path(args.output_dir).is_dir() else None
    print("reanudando desde", ultimo) if ultimo else print("empezando desde cero")
    trainer.train(resume_from_checkpoint=ultimo)
    trainer.save_model(str(FINAL))            # = el mejor checkpoint según WER en dev
    processor.save_pretrained(str(FINAL))
    hist = pd.DataFrame(trainer.state.log_history)
    hist.to_csv(out / "historial.csv", index=False)
    print("mejor checkpoint:", trainer.state.best_model_checkpoint, "| WER dev:", trainer.state.best_metric)

hist = pd.read_csv(out / "historial.csv")
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
for col, nombre in (("loss", "train"), ("eval_loss", "dev")):
    if col in hist.columns:
        h = hist.dropna(subset=[col]); ax[0].plot(h.step, h[col], label=nombre)
ax[0].set_title("loss"); ax[0].legend()
if "eval_wer" in hist.columns:
    e = hist.dropna(subset=["eval_wer"]); ax[1].plot(e.step, e.eval_wer, marker="o")
ax[1].set_title("WER dev (%)")
plt.tight_layout(); plt.savefig(out / "curvas.png", dpi=120); plt.show()

## 9. Evaluación final (mejor checkpoint)

WER de corpus con IC 95% por bootstrap sobre clips. Las predicciones por clip quedan en `pred_*.csv` para el análisis de errores.

In [ ]:
model.to(device).eval()
res = {}
for nombre, df in {"dev": dev_df, **EVAL}.items():
    pred = C.transcribir(model, processor, df, RAIZ, device)
    pred.to_csv(out / f"pred_{nombre}.csv", index=False)
    res[nombre] = C.wer_con_ic(pred)
(out / "resultados.json").write_text(json.dumps(res, indent=2))

zs = json.loads(f_zs.read_text()) if f_zs.exists() else {}
tabla = pd.DataFrame({n: {"WER fine-tuned": f"{r['wer']} [{r['ic95'][0]}–{r['ic95'][1]}]", "CER": r["cer"],
                          "WER zero-shot": zs.get(n, {}).get("wer", "-"), "clips": r["clips"],
                          "palabras": r["palabras"]} for n, r in res.items()}).T
print(tabla.to_string())

## 10. Escuchar ejemplos y descargar

Tres clips de voces no vistas: lo que dice el hablante y lo que transcribe el modelo.

In [ ]:
import IPython.display as ipd
p = pd.read_csv(out / "pred_test_otros.csv").sample(3, random_state=SEED)
for r in p.itertuples():
    print(f"{r.id}  (WER {r.wer_clip}%)\n  real  : {r.text_norm}\n  modelo: {r.prediccion}")
    ipd.display(ipd.Audio(str(RAIZ / r.file_name)))

if not DRIVE_OK:   # sin Drive: empaquetar lo importante para bajarlo
    with zipfile.ZipFile("/content/resultados_asr.zip", "w") as z:   # sin checkpoints (GB de optimizador)
        for f in out.rglob("*"):
            if f.is_file() and "checkpoints" not in f.parts:
                z.write(f, f.relative_to(out.parent))
    print("descarga /content/resultados_asr.zip desde el panel de archivos")